In [53]:
import pandas as pd


In [54]:
df = pd.read_csv("data/out_cleaned.csv", sep=";")


In [55]:
print(df.dtypes)


timestamp                int64
CN.VIEWS3600           float64
CN.VIEWS7200           float64
CN.VIEWS10800          float64
CN.VIEWS14400          float64
CN.VIEWS18000          float64
CN.VIEWS21600          float64
CN.VIEWS25200          float64
CN.VIEWS28800          float64
CN.VIEWS32400          float64
CN.VIEWS36000          float64
CN.VIEWS39600          float64
CN.VIEWS43200          float64
CN.VIEWS46800          float64
CN.VIEWS50400          float64
CN.VIEWS54000          float64
CN.VIEWS57600          float64
CN.VIEWS61200          float64
CN.VIEWS64800          float64
CN.VIEWS68400          float64
CN.VIEWS72000          float64
CN.VIEWS75600          float64
CN.VIEWS79200          float64
CN.VIEWS82800          float64
CN.VIEWS86400          float64
CN.VIEWS604800         float64
CN.VIEWS3600_DERIV     float64
CN.VIEWS7200_DERIV     float64
CN.VIEWS10800_DERIV    float64
CN.VIEWS14400_DERIV    float64
CN.VIEWS18000_DERIV    float64
CN.VIEWS21600_DERIV    float64
CN.VIEWS

In [56]:
class c:
    timestamp = "timestamp"
    views_3 = "CN.VIEWS10800"
    views_6 = "CN.VIEWS21600"
    views_9 = "CN.VIEWS32400"
    views_12 = "CN.VIEWS43200"

    target = "target"


# <72M
# 72–73М
# 73–74 млн
# 74–75 млн
# 75–76 млн
# 76М+


In [57]:
target_col = "CN.VIEWS604800"
target_value = 70000000

df[c.target] = df[target_col] > target_value
print(df[c.target].sum() / len(df[c.target]))

0.2


In [58]:
columns = [c.timestamp, c.views_3, c.views_6, c.views_9, c.views_12, c.target]


# df[c.target] = df["CN.VIEWS86400"]

df.drop(columns=[col for col in df.columns if col not in columns], inplace=True)

df[c.views_6] = df[c.views_6] - df[c.views_3]
df[c.views_9] = df[c.views_9] - df[c.views_6]
df[c.views_12] = df[c.views_12] - df[c.views_9]

df.sort_values(by=c.timestamp, inplace=True)

df.head()


,timestamp,CN.VIEWS10800,CN.VIEWS21600,CN.VIEWS32400,CN.VIEWS43200,target
203,1529793896,276775.544476,355704.236677,5.418991e+05,556129.399307,False
189,1529880698,338803.498204,272207.016319,4.952770e+05,416134.506949,False
124,1530226341,351442.008331,328092.000399,5.788078e+05,519775.669048,False
155,1530398253,539806.426075,650288.124326,9.442201e+05,946476.273086,False
12,1531349885,571956.430689,541626.197976,1.122434e+06,829318.108365,False


In [59]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Разделение данных
X = df[[c.views_3, c.views_6, c.views_9, c.views_12]]
y = df[c.target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

scale_pos_weight = y_train.value_counts()[False] / y_train.value_counts()[True]

# Создание и обучение модели
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbosity=1,
)

model.fit(X_train, y_train)

# Предсказание
y_pred = model.predict(X_test)

print(X_train[:10])
print(X_test[:2])
print(y_test[:2])
print(model.predict(X_test[:2]))

# Оценка
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(len(y_train))

     CN.VIEWS10800  CN.VIEWS21600  CN.VIEWS32400  CN.VIEWS43200
203   2.767755e+05   3.557042e+05   5.418991e+05   5.561294e+05
189   3.388035e+05   2.722070e+05   4.952770e+05   4.161345e+05
124   3.514420e+05   3.280920e+05   5.788078e+05   5.197757e+05
155   5.398064e+05   6.502881e+05   9.442201e+05   9.464763e+05
12    5.719564e+05   5.416262e+05   1.122434e+06   8.293181e+05
79    1.285919e+06   1.214239e+06   2.123381e+06   1.904937e+06
196   4.166593e+05   7.792568e+05   8.579332e+05   1.100446e+06
0     6.200204e+05   6.348049e+05   1.109465e+06   1.017201e+06
160   5.658962e+05   4.717088e+05   9.610097e+05   7.308032e+05
19    6.908407e+05   7.339038e+05   1.168502e+06   1.101787e+06
     CN.VIEWS10800  CN.VIEWS21600  CN.VIEWS32400  CN.VIEWS43200
112   9.359413e+06   8.487473e+06   1.576036e+07   1.454715e+07
110   9.759134e+06   8.836990e+06   1.588670e+07   1.452231e+07
112    True
110    True
Name: target, dtype: bool
[1 1]
0.8444444444444444
              precision    re

In [60]:
from sklearn.metrics import confusion_matrix

# Анализ дисбаланса
print("Реальное распределение:")
print(y_test.value_counts())

print("\nПредсказания модели:")
print(pd.Series(y_pred).value_counts())

# Матрица ошибок
cm = confusion_matrix(y_test, y_pred)
print("\nМатрица ошибок:")
print(f"True Negatives (правильно False): {cm[0, 0]}")
print(f"False Positives (неправильно True): {cm[0, 1]}")
print(f"False Negatives (неправильно False): {cm[1, 0]}")
print(f"True Positives (правильно True): {cm[1, 1]}")

Реальное распределение:
target
True     33
False    12
Name: count, dtype: int64

Предсказания модели:
1    38
0     7
Name: count, dtype: int64

Матрица ошибок:
True Negatives (правильно False): 6
False Positives (неправильно True): 6
False Negatives (неправильно False): 1
True Positives (правильно True): 32


In [61]:
# Получите вероятности вместо 0/1
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Тестирование разных порогов
thresholds = [0.5, 0.55, 0.6, 0.65, 0.7]

for threshold in thresholds:
    y_pred_adjusted = (y_pred_proba >= threshold).astype(int)

    print(f"\n{'=' * 50}")
    print(f"Порог: {threshold}")
    print(f"{'=' * 50}")

    print("\nРаспределение предсказаний:")
    print(pd.Series(y_pred_adjusted).value_counts())

    print("\nМетрики:")
    print(classification_report(y_test, y_pred_adjusted))

    cm = confusion_matrix(y_test, y_pred_adjusted)
    print(f"True Negatives: {cm[0, 0]}, False Positives: {cm[0, 1]}")
    print(f"False Negatives: {cm[1, 0]}, True Positives: {cm[1, 1]}")


Порог: 0.5

Распределение предсказаний:
1    38
0     7
Name: count, dtype: int64

Метрики:
              precision    recall  f1-score   support

       False       0.86      0.50      0.63        12
        True       0.84      0.97      0.90        33

    accuracy                           0.84        45
   macro avg       0.85      0.73      0.77        45
weighted avg       0.85      0.84      0.83        45

True Negatives: 6, False Positives: 6
False Negatives: 1, True Positives: 32

Порог: 0.55

Распределение предсказаний:
1    38
0     7
Name: count, dtype: int64

Метрики:
              precision    recall  f1-score   support

       False       0.86      0.50      0.63        12
        True       0.84      0.97      0.90        33

    accuracy                           0.84        45
   macro avg       0.85      0.73      0.77        45
weighted avg       0.85      0.84      0.83        45

True Negatives: 6, False Positives: 6
False Negatives: 1, True Positives: 32

Поро

In [63]:
# Применить оптимальный порог
OPTIMAL_THRESHOLD = 0.9  # Выберите нужный порог

# Получите новые предсказания с повышенным порогом
y_pred_final = (y_pred_proba >= OPTIMAL_THRESHOLD).astype(int)

print(f"Порог вероятности: {OPTIMAL_THRESHOLD}")
print(f"\n{'=' * 50}")
print("ИТОГОВЫЕ МЕТРИКИ:")
print(f"{'=' * 50}\n")

print("Распределение предсказаний:")
print(f"True: {y_pred_final.sum()}")
print(f"False: {(1 - y_pred_final).sum()}\n")

print(classification_report(y_test, y_pred_final))

# Новая матрица ошибок
cm_final = confusion_matrix(y_test, y_pred_final)
print(f"True Negatives (правильно False): {cm_final[0, 0]}")
print(f"False Positives (неправильно True): {cm_final[0, 1]}")
print(f"False Negatives (неправильно False): {cm_final[1, 0]}")
print(f"True Positives (правильно True): {cm_final[1, 1]}")

Порог вероятности: 0.9

ИТОГОВЫЕ МЕТРИКИ:

Распределение предсказаний:
True: 38
False: 7

              precision    recall  f1-score   support

       False       0.86      0.50      0.63        12
        True       0.84      0.97      0.90        33

    accuracy                           0.84        45
   macro avg       0.85      0.73      0.77        45
weighted avg       0.85      0.84      0.83        45

True Negatives (правильно False): 6
False Positives (неправильно True): 6
False Negatives (неправильно False): 1
True Positives (правильно True): 32
